# 🤖 Módulo 03 — Machine Learning: A IA Descobre Padrões Médicos Sozinha
### Minicurso: Informática Biomédica Aplicada | Jornada InfoBio 2026

---

## Você chegou ao Machine Learning! 🚀🏆

Este é o módulo mais avançado da Jornada InfoBio 2026. O fato de você estar aqui significa que você já sabe **visualizar dados biomédicos** e **normalizar escalas** — duas habilidades fundamentais em Ciência de Dados na Saúde.

Agora vamos usar um algoritmo real de inteligência artificial, o **K-Means**, para descobrir padrões fisiológicos nos nossos atletas — **sem que ninguém tenha dito ao algoritmo o que procurar**.

Esse tipo de análise, chamada de **aprendizado não-supervisionado**, é usada em hospitais e centros esportivos ao redor do mundo para identificar grupos de risco antes que o atleta sinta qualquer sintoma.

---

## 🧠 O que é K-Means?

K-Means é um algoritmo que recebe uma pergunta simples:

> *"Dada uma nuvem de pontos, divida-a em K grupos. Cada ponto deve pertencer ao grupo cujo centro está mais próximo."*

Nós escolhemos **K = 3**, que no contexto esportivo pode representar:

| Grupo | Perfil Fisiológico | Decisão da Comissão Técnica |
|---|---|---|
| 🟢 Grupo 1 | Recuperado — marcadores normais | Pode jogar normalmente |
| 🟡 Grupo 2 | Fatigado — dano moderado | Treino leve, monitorar |
| 🔴 Grupo 3 | Alto Risco — estresse severo | Repouso obrigatório |

O algoritmo não sabe que esses grupos existem. Ele os **descobre sozinho** a partir dos biomarcadores.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Carregando e normalizando (notebook self-contained)
df = pd.read_csv('dados_atletas_minicurso.csv')

colunas_bio = ['CK_UL', 'Cortisol_ugdL', 'LDH_UL', 'PCR_mgL', 'Testosterona_nmolL']
for col in colunas_bio:
    df[col + '_z'] = (df[col] - df[col].mean()) / df[col].std()

colunas_z = [c + '_z' for c in colunas_bio]
X = df[colunas_z].values

print('Dados preparados!')
print(f'Matriz de entrada para o K-Means: {X.shape[0]} atletas × {X.shape[1]} biomarcadores normalizados')

In [ ]:
# Rodando o K-Means com K=3
# random_state=42 garante que você obtenha o mesmo resultado toda vez que rodar
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X)

print('K-Means concluído! Distribuição dos atletas por cluster:')
print(df['Cluster'].value_counts().sort_index().rename({0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2'}))
print()
df[['ID_Atleta', 'Posicao', 'CK_UL', 'Cortisol_ugdL', 'Cluster']].sort_values('Cluster')

In [ ]:
plt.figure(figsize=(10, 7))

cores = {0: '#2196F3', 1: '#FF9800', 2: '#4CAF50'}
marcadores_cluster = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2'}

for cluster_id in sorted(df['Cluster'].unique()):
    subset = df[df['Cluster'] == cluster_id]
    plt.scatter(
        subset['CK_UL'],
        subset['Cortisol_ugdL'],
        c=cores[cluster_id],
        label=marcadores_cluster[cluster_id],
        s=140,
        edgecolors='white',
        linewidths=1.2,
        alpha=0.9
    )

plt.title('K-Means (k=3) — Agrupamento Fisiológico dos Atletas', fontsize=14, fontweight='bold')
plt.xlabel('CK — Creatina Quinase (U/L)\n[Dano Muscular Mecânico]', fontsize=11)
plt.ylabel('Cortisol (µg/dL)\n[Estresse Sistêmico]', fontsize=11)
plt.legend(title='Grupos Descobertos pela IA', fontsize=10)
plt.tight_layout()
plt.show()

## 🔬 O que a IA descobriu?

Cada cor no gráfico é um **estado fisiológico diferente** que o algoritmo identificou automaticamente, analisando os 5 biomarcadores **ao mesmo tempo**.

Podemos explorar os centroides (o "átleta médio" de cada grupo) para interpretar o que cada cluster representa:

- Um cluster com **CK alta** representa atletas com **dano muscular mecânico intenso** — o músculo foi muito exigido.
- Um cluster com **Cortisol alto** representa atletas em **estresse sistêmico elevado** — o sistema hormonal está em alerta.
- Um cluster com marcadores **próximos da média** representa atletas em estado de **recuperação normal**.

### O conceito de "Risco Silencioso"

No artigo original (Rosito et al., 2026), os pesquisadores identificaram um padrão particularmente preocupante:

> *Alguns atletas apresentavam **Cortisol muito elevado**, mas **CK completamente normal**.*

Na análise tradicional — que olha um marcador de cada vez — esses atletas **passariam despercebidos**: a CK está normal, então parece que está tudo bem.

Mas o modelo multivariado consegue detectar esse padrão: **estresse sistêmico severo sem dano mecânico visível**. Esse é o chamado **"Risco Silencioso"** — invisível na análise univariada, mas claramente identificado quando analisamos todos os biomarcadores juntos.

Olhe o scatter plot acima: você consegue identificar um atleta que está no quadrante de **Cortisol alto + CK baixo**? Esses são os candidatos ao perfil de Risco Silencioso.

In [ ]:
# Perfil médio de cada cluster (Z-Score) — o 'átleta típico' de cada grupo
centroides = df.groupby('Cluster')[colunas_z].mean().round(2)
centroides.index = ['Cluster 0', 'Cluster 1', 'Cluster 2']
centroides.columns = ['CK', 'Cortisol', 'LDH', 'PCR', 'Testosterona']

print('Perfil médio de cada cluster (valores em Z-Score):')
print('Positivo (+) = acima da média do grupo | Negativo (−) = abaixo da média')
print()

plt.figure(figsize=(9, 4))
sns.heatmap(
    centroides,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn_r',
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Z-Score'}
)
plt.title('Heatmap dos Centroides — Perfil Fisiológico de Cada Cluster', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 🏆 DESAFIO DO BOMBOM MASTER 🏆

O instrutor irá revelar as regras e os critérios deste desafio no telão. Prepare o seu código!

In [ ]:
# ✏️ SEU CÓDIGO AQUI
